# Hit rate analyses - testing the hallucination rates for each condition

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_1samp

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pingouin as pg

In [20]:
#Defining project root
import sys
from pathlib import Path
# Make sure we can import config.py from project root

# One folder up from current notebook location
project_root = Path.cwd().parent.parent.resolve()

# Add subdirectories to path
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'config'))
print(f"Project root: {project_root}")
import config

Project root: /mnt/hdd/anatkorol/Imagination_in_translation


In [30]:
df = pd.read_csv("/mnt/hdd/anatkorol/Imagination_in_translation/Data/processed_data/29032026_pilot_2_perception_no_feedback/nlp_analysis/ppt_trials_w_object_validation.csv").copy()
condition = "perception_no_feedback" # change this according to the condition you want to analyze, e.g. "perception_no_feedback", "gpt-5_descriptions_as_ppt", "translation_imagination"

In [31]:
df.columns

Index(['uid', 'gt', 'session', 'attempt', 'prompt', 'gen', 'subjective_score',
       'prompt_latency_secs', 'generating_latency_secs', 'rating_latency_secs',
       'ts', 'study_result', 'comp_result', 'token_num', 'extraction',
       'objects', 'attr_color', 'attr_shape', 'attr_size', 'attr_material',
       'attr_texture', 'attr_pose', 'attr_action', 'attr_state',
       'spatial_relations', 'world_knowledge', 'scene', 'camera_aspects',
       'optical_effects', 'subjective_detail', 'item_evaluations',
       'validated_objects', 'invalid_not_objects', 'invalid_not_in_image',
       'n_extracted', 'n_validated', 'n_invalid_not_object',
       'n_invalid_not_in_image', 'diff_in_objects', 'error'],
      dtype='object')

In [32]:
# -----------------------------
# Executive summary - for the whole csv
# -----------------------------
summary_df = df
print(f"\n=== FULL DATA SUMMARY for {condition}===")
# Coerce numeric columns in case some rows failed and produced None / strings
numeric_cols = [
    "n_extracted",
    "n_validated",
    "n_invalid_not_object",
    "n_invalid_not_in_image",
    "diff_in_objects",
]
for col in numeric_cols:
    summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

n_rows = len(summary_df)
n_error_rows = summary_df["error"].notna().sum() if "error" in summary_df.columns else 0
n_success_rows = n_rows - n_error_rows

total_extracted = summary_df["n_extracted"].sum(skipna=True)
total_validated = summary_df["n_validated"].sum(skipna=True)
total_semantic_errors = summary_df["n_invalid_not_object"].sum(skipna=True)
total_hallucinations = summary_df["n_invalid_not_in_image"].sum(skipna=True)
total_invalid_combined = summary_df["diff_in_objects"].sum(skipna=True)
# Safe percentage helper
def pct(num, den):
    return (100 * num / den) if den and den > 0 else 0.0

semantic_error_pct = pct(total_semantic_errors, total_extracted)
hallucination_pct = pct(total_hallucinations, total_extracted)
valid_pct = pct(total_validated, total_extracted)
combined_invalid_pct = pct(total_invalid_combined, total_extracted)

rows_with_hallucination = (summary_df["n_invalid_not_in_image"].fillna(0) > 0).sum()
rows_with_semantic_error = (summary_df["n_invalid_not_object"].fillna(0) > 0).sum()
rows_with_any_invalid = (summary_df["diff_in_objects"].fillna(0) > 0).sum()

avg_extracted_per_row = summary_df["n_extracted"].mean(skipna=True)
avg_validated_per_row = summary_df["n_validated"].mean(skipna=True)
avg_semantic_errors_per_row = summary_df["n_invalid_not_object"].mean(skipna=True)
avg_hallucinations_per_row = summary_df["n_invalid_not_in_image"].mean(skipna=True)

executive_summary = {
    "n_rows": int(n_rows),
    "n_success_rows": int(n_success_rows),
    "n_error_rows": int(n_error_rows),

    "total_extracted_objects": int(total_extracted),
    "total_validated_objects": int(total_validated),

    "total_semantic_errors": int(total_semantic_errors),
    "semantic_error_pct_of_all_extracted_objects": round(semantic_error_pct, 2),

    "total_hallucinations": int(total_hallucinations),
    "hallucination_pct_of_all_extracted_objects": round(hallucination_pct, 2),

    "total_invalid_combined": int(total_invalid_combined),
    "combined_invalid_pct_of_all_extracted_objects": round(combined_invalid_pct, 2),

    "valid_object_pct_of_all_extracted_objects": round(valid_pct, 2),

    "rows_with_at_least_one_hallucination": int(rows_with_hallucination),
    "rows_with_at_least_one_semantic_error": int(rows_with_semantic_error),
    "rows_with_at_least_one_invalid_item": int(rows_with_any_invalid),

    "avg_extracted_objects_per_row": round(avg_extracted_per_row, 3) if pd.notna(avg_extracted_per_row) else None,
    "avg_validated_objects_per_row": round(avg_validated_per_row, 3) if pd.notna(avg_validated_per_row) else None,
    "avg_semantic_errors_per_row": round(avg_semantic_errors_per_row, 3) if pd.notna(avg_semantic_errors_per_row) else None,
    "avg_hallucinations_per_row": round(avg_hallucinations_per_row, 3) if pd.notna(avg_hallucinations_per_row) else None,
}

print("\n=== EXECUTIVE SUMMARY ===")
for k, v in executive_summary.items():
    print(f"{k}: {v}")



=== FULL DATA SUMMARY for perception_no_feedback===

=== EXECUTIVE SUMMARY ===
n_rows: 90
n_success_rows: 90
n_error_rows: 0
total_extracted_objects: 731
total_validated_objects: 561
total_semantic_errors: 127
semantic_error_pct_of_all_extracted_objects: 17.37
total_hallucinations: 45
hallucination_pct_of_all_extracted_objects: 6.16
total_invalid_combined: 170
combined_invalid_pct_of_all_extracted_objects: 23.26
valid_object_pct_of_all_extracted_objects: 76.74
rows_with_at_least_one_hallucination: 31
rows_with_at_least_one_semantic_error: 73
rows_with_at_least_one_invalid_item: 80
avg_extracted_objects_per_row: 8.122
avg_validated_objects_per_row: 6.233
avg_semantic_errors_per_row: 1.411
avg_hallucinations_per_row: 0.5


## Now let's look at the hallucinations themselves
### Which rows have hallucinations
### Which hallucinated objects are most common
### A manual review view with image + prompt + hallucinated items

In [36]:
#filter only rows with hallucinations (n_invalid_not_in_image > 0)
hall_df = df[df["n_invalid_not_in_image"] > 0].copy()
print(len(hall_df))
hall_df[["prompt", "objects", "invalid_not_in_image", "n_invalid_not_in_image"]].head()

31


,prompt,objects,invalid_not_in_image,n_invalid_not_in_image
2,"A modern, stylish livingnroom in blues with ma...","['livingroom', 'rug', 'coffee table', 'sofa', ...","[""picture""]",1
5,An old lighthouse that looks like it's sinking...,"['lighthouse', 'waters', 'window', 'door', 'li...","[""light""]",1
7,A warm inviting bedroom in earth tones with so...,"['bedroom', 'bed', 'picture', 'frame', 'duvet'...","[""basket""]",1
11,An old white clocktower with blue sky in the b...,"['clocktower', 'sky', 'tree', 'building', 'clo...","[""clock face""]",1
14,A modern meeting room with a long square table...,"['meeting room', 'table', 'chair', 'opening', ...","[""desk""]",1


In [39]:
pd.set_option("display.max_colwidth", None)
hall_df[["prompt", "objects", "invalid_not_in_image", "n_invalid_not_in_image"]]

,prompt,objects,invalid_not_in_image,n_invalid_not_in_image
2,"A modern, stylish livingnroom in blues with many accented colored circular rugs.The rugs are 3 foot circular rugs that are yellow, pink, green, white and royal blue. There is a coffee table in front of the four person, royal blue sofa with minimalist vase with yellow flowers. There is a very large mural/picture in blue tones that is the length of the very long couch.","['livingroom', 'rug', 'coffee table', 'sofa', 'vase', 'flower', 'mural', 'picture']","[""picture""]",1
5,"An old lighthouse that looks like it's sinking in dark blue waters. The lighthouse is weathered and rusted and looks easily over 100 years old. It's listing to the right and you can see a window and a door that shows the scale that it's actually a large lighthouse, you can see the light portion at the top of the turret.","['lighthouse', 'waters', 'window', 'door', 'light', 'turret']","[""light""]",1
7,A warm inviting bedroom in earth tones with some color accents and a huge modern 4 poster bed. Two nature pictures with brown frames hang over the bed. The bed is beautifully made with a tufted duvet and pale yellow accent pillows. At the foot of the bed is a bench with a basket full of books.,"['bedroom', 'bed', 'picture', 'frame', 'duvet', 'pillow', 'bench', 'basket', 'book']","[""basket""]",1
11,"An old white clocktower with blue sky in the background and surrounded by trees. The trees, which look full grown, don't reach the halfway point of the tower, which indicates it's very very tall. There is a modern building to the left of the tower, which is maybe an apartment building or an office buiding and has many stories and contrasts with the ornate nature of the clock tower. Although the modern building has many stories, it only reaches a third up the tower. The clock faces, showing on two sides that we can see, looks like it's 11:30, so it's almost noon a beautiful summer day. There are three high narrow openings on each side at the top of the tower, which might mean it's not only a clock tower but a bell tower as well.","['clocktower', 'sky', 'tree', 'building', 'clock face', 'opening']","[""clock face""]",1
14,"A modern meeting room with a long square table that's shaped in an open square with four bright red chairs on each of 3 sides. There is one opening in the large square which looks like an area where someone can enter the center and lead a team meeting. There are also 3 large posters/diagrams and two white boards. The atmosphere is bright and efficient and there's a large tall potted plant in the corner. The room is very tidy but the chairs are not perfectly pushed in or lined up, and there's a piece of paper on one of the long desks, as if the room is taking a break. There are white vertical blinds on the long large wall of windows on the opposite side of the room.","['meeting room', 'table', 'chair', 'opening', 'poster', 'diagram', 'white board', 'potted plant', 'corner', 'piece of paper', 'desk', 'blinds', 'wall', 'window']","[""desk""]",1
18,"This photo takes place indoors. The lighting appears to be somewhat dim. The walls are painted an eggshell white color. On the upper right hand side of the image there are two white boards. On the upper left hand side of the image, white window appear to be slightly open. The floors in the room appear to white linoleum. In the center of the image there is a very large wrap-around table that covers much of the room. This table is in the shape of a square, and the middle is hallow. In center of the photo near the top it appears as though there is a small opening in the desk structure. The tabletop is a a light, natural maple color. There are white. Surrounding this wrap-around desk are many bright red chairs with slender black legs. In this photo, approximately ten chairs in total are visible.","['wall', 'white board', 'window', 'floor', 'table', 'desk', 'chair']","[""desk""]",1
26,"This photo takes place indoors. The image is a bedroom. The walls 